# A posterior on a grid

A model made $k$ errors on $n$ test examples, and its error rate $p$ is unknown. With a Beta
prior the posterior is another Beta curve, but a team's real prior is often some other shape.
Here the team is sure, from earlier models of the same kind, that the error rate is below 0.2,
and is otherwise undecided. No Beta curve says that. A grid handles any prior: cut $[0, 1]$
into 1,000 small cells, give each cell's centre a weight proportional to prior times
likelihood, and scale the weights to sum to 1.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# The centres of 1,000 equal cells covering [0, 1].
p = (np.arange(1000) + 0.5) / 1000
flat = np.ones_like(p)                 # every error rate equally plausible
capped = (p < 0.2).astype(float)       # sure the rate is below 0.2, undecided below that
print(p[:3], p[-3:])

## Part 1 (coding task): Posterior weights

Write `grid_posterior(prior, p, k, n)`. It returns an array `w` of the same shape as `p`, with
every entry at least 0 and the entries summing to 1, where `w[i]` is proportional to

$$
\text{prior}[i] \cdot p_i^{\,k} (1 - p_i)^{n - k} .
$$

It must also work when $n$ is in the thousands. Then $p^k(1 - p)^{n-k}$ can be 0 at every grid
point, because it is smaller than the smallest number a float can hold. Work with logarithms
instead, and subtract the largest log weight before you exponentiate. A prior entry of 0 has
logarithm $-\infty$, which `np.exp` turns back into 0.

In [ ]:
def grid_posterior(prior, p, k, n):
    """Weights on the grid p, summing to 1, proportional to prior * p**k * (1 - p)**(n - k)."""
    # TODO
    raise NotImplementedError

In [ ]:
w = grid_posterior(flat, p, 3, 10)
assert w.shape == p.shape, f"expected shape {p.shape}, got {w.shape}"
assert np.all(w >= 0) and np.isclose(w.sum(), 1), "the weights should be at least 0 and sum to 1"
mean = np.sum(p * w)
assert abs(mean - 1 / 3) < 1e-4, f"posterior mean {mean:.4f}: a flat prior and 3 errors in 10 give Beta(4, 8), whose mean is 1/3"
w_cap = grid_posterior(capped, p, 3, 10)
assert np.isclose(w_cap[p >= 0.2].sum(), 0), "the capped prior is 0 from 0.2 up, so the posterior must be 0 there too"
big = grid_posterior(flat, p, 600, 2000)
assert np.all(np.isfinite(big)), "with n = 2000 the plain product underflows to 0 everywhere: compute the weights in logs"
assert np.isclose(big.sum(), 1), "the weights should sum to 1"
assert abs(np.sum(p * big) - 601 / 2002) < 1e-4, "600 errors in 2,000 with a flat prior give Beta(601, 1401), mean 601/2002"
print("Part 1 passed")

## Part 2 (coding task): Summaries of the posterior

Write `summaries(p, w, t)`, which takes a grid and its weights and returns three numbers:

- `mean`: the posterior average of the error rate, each grid value weighted by its weight.
- `above`: the posterior probability that the error rate is greater than `t`.
- `upper`: the smallest grid value $u$ such that the weights of the grid values up to and
  including $u$ add up to at least 0.95. The team can then say the error rate is at most $u$
  with probability 0.95.

`np.cumsum` adds weights up from the left.

In [ ]:
def summaries(p, w, t):
    """Return (mean, above, upper) for the grid p with weights w."""
    # TODO
    raise NotImplementedError

In [ ]:
w = grid_posterior(flat, p, 3, 10)
mean, above, upper = summaries(p, w, 0.5)
assert abs(mean - 1 / 3) < 1e-4, f"mean {mean:.4f}: weight each grid value by its posterior weight"
assert abs(above - 232 / 2048) < 5e-4, f"above {above:.4f}: under Beta(4, 8), P(p > 0.5) = 232/2048, about 0.1133"
assert abs(upper - 0.5644) < 5e-4, f"upper {upper:.4f}: the 0.95 point of Beta(4, 8) is about 0.564"
w_cap = grid_posterior(capped, p, 3, 10)
m_cap, a_cap, u_cap = summaries(p, w_cap, 0.15)
assert abs(m_cap - 0.1501) < 1e-3, f"mean {m_cap:.4f} with the capped prior: expected about 0.150"
assert abs(a_cap - 0.5690) < 5e-4, f"above {a_cap:.4f} with the capped prior: expected about 0.569"
assert abs(u_cap - 0.1963) < 5e-4, f"upper {u_cap:.4f} with the capped prior: expected about 0.196"
print("Part 2 passed")
print(f"flat prior:   mean {mean:.3f}, P(p > 0.5) {above:.3f}, 95% upper bound {upper:.3f}")
print(f"capped prior: mean {m_cap:.3f}, P(p > 0.15) {a_cap:.3f}, 95% upper bound {u_cap:.3f}")

## Solution

### Part 1: Posterior weights

In [ ]:
def grid_posterior(prior, p, k, n):
    """Weights on the grid p, summing to 1, proportional to prior * p**k * (1 - p)**(n - k)."""
    with np.errstate(divide="ignore"):         # a zero prior becomes minus infinity, then 0 after exp
        log_w = np.log(prior) + k * np.log(p) + (n - k) * np.log1p(-p)
    log_w = log_w - log_w.max()                # the largest weight becomes exp(0) = 1
    w = np.exp(log_w)
    return w / w.sum()

**Why it works.** Taking logs turns the product into a sum, $\log \text{prior} + k \log p + (n - k)\log(1 - p)$,
and a sum of a few thousand moderate terms is an ordinary float. Subtracting the largest log
weight multiplies every weight by the same constant, which the final division removes anyway,
and it makes the largest weight exactly 1, so `np.exp` cannot underflow everywhere at once.
`np.log1p(-p)` computes $\log(1 - p)$ accurately when $p$ is small. The zero prior above 0.2
gives $-\infty$, and `np.exp(-np.inf)` is 0, so those cells get no weight.

### Part 2: Summaries of the posterior

In [ ]:
def summaries(p, w, t):
    """Return (mean, above, upper) for the grid p with weights w."""
    mean = np.sum(p * w)
    above = w[p > t].sum()
    upper = p[np.searchsorted(np.cumsum(w), 0.95)]   # first index where the running total reaches 0.95
    return mean, above, upper

**Why it works.** The weights are the posterior probabilities of the grid cells, so a probability is a sum of
weights and an average is a weighted sum. `np.cumsum(w)` gives the posterior probability of
$p \le p_i$ at each grid point, and `np.searchsorted` finds the first point where that total
reaches 0.95. With 3 errors in 10, the flat prior allows rates up to 0.56 at the 0.95 level.
The capped prior keeps every rate below 0.2, and the data push its posterior against that
cap: the mean is 0.15 and the 0.95 bound about 0.196, where the raw rate was 0.3.